# MLOps Toolkit Demo

Full walkthrough: **Train model → Register → Serve → Monitor → A/B Test**

## Prerequisites

```bash
# 1. Train example models
python scripts/train_example_model.py

# 2. Start the full stack
docker compose up --build -d
```

In [ ]:
import requests
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt

BASE_URL = "http://localhost:8000"

def api(method, path, **kwargs):
    resp = getattr(requests, method)(f"{BASE_URL}{path}", **kwargs)
    resp.raise_for_status()
    return resp.json()

## 1. Health Check & Model Info

In [ ]:
health = api("get", "/health")
print("Health:", json.dumps(health, indent=2))

info = api("get", "/model/info")
print("\nModel Info:", json.dumps(info, indent=2))

## 2. Run Predictions

Using Iris dataset features (sepal length, sepal width, petal length, petal width).

In [ ]:
# Activate model v1
api("post", "/model/activate", json={"version": "v1"})

# Test samples from each Iris class
test_samples = [
    ([5.1, 3.5, 1.4, 0.2], "setosa"),
    ([7.0, 3.2, 4.7, 1.4], "versicolor"),
    ([6.3, 3.3, 6.0, 2.5], "virginica"),
]
iris_classes = ["setosa", "versicolor", "virginica"]

print("Predictions with v1 (LogisticRegression):")
for features, expected in test_samples:
    result = api("post", "/predict", json={"features": features})
    predicted = iris_classes[int(result["prediction"])]
    print(f"  {features} -> {predicted} (conf={result['confidence']:.4f}, latency={result['latency_ms']:.2f}ms)")

## 3. Compare Model Versions

In [ ]:
# Compare v1 vs v2 on same inputs
random.seed(42)
test_features = [[round(random.uniform(4, 8), 1), round(random.uniform(2, 4.5), 1),
                   round(random.uniform(1, 7), 1), round(random.uniform(0.1, 2.5), 1)]
                  for _ in range(50)]

results = {}
for version in ["v1", "v2"]:
    api("post", "/model/activate", json={"version": version})
    v_results = []
    for features in test_features:
        r = api("post", "/predict", json={"features": features, "model_version": version})
        v_results.append(r)
    results[version] = v_results

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for version, color in [("v1", "steelblue"), ("v2", "coral")]:
    confs = [r["confidence"] for r in results[version]]
    lats = [r["latency_ms"] for r in results[version]]
    axes[0].hist(confs, bins=15, alpha=0.6, label=version, color=color)
    axes[1].hist(lats, bins=15, alpha=0.6, label=version, color=color)

axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("Count")
axes[0].set_title("Prediction Confidence Distribution"); axes[0].legend()
axes[1].set_xlabel("Latency (ms)"); axes[1].set_ylabel("Count")
axes[1].set_title("Prediction Latency Distribution"); axes[1].legend()
plt.tight_layout()
plt.show()

## 4. A/B Testing

In [ ]:
# Configure A/B test: 70% v1, 30% v2
config = api("post", "/ab-test/configure", json={
    "model_a_version": "v1",
    "model_b_version": "v2",
    "traffic_split": 0.7,
    "enabled": True
})
print("A/B Config:", json.dumps(config, indent=2))

# Send 200 requests through A/B router
print("\nSending 200 requests through A/B router...")
ab_predictions = []
for features in test_features * 4:  # 200 requests
    r = api("post", "/predict", json={"features": features})
    ab_predictions.append(r)

# Get results
ab_results = api("get", "/ab-test/results")
print("\nA/B Test Results:")
print(json.dumps(ab_results, indent=2))

In [ ]:
# Visualize A/B test results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Traffic split
axes[0].bar(["Model A (v1)", "Model B (v2)"],
            [ab_results["model_a_count"], ab_results["model_b_count"]],
            color=["steelblue", "coral"])
axes[0].set_title("Traffic Distribution")
axes[0].set_ylabel("Request Count")

# Average confidence
axes[1].bar(["Model A (v1)", "Model B (v2)"],
            [ab_results["model_a_avg_confidence"], ab_results["model_b_avg_confidence"]],
            color=["steelblue", "coral"])
axes[1].set_title("Average Confidence")
axes[1].set_ylim(0, 1)

# Average latency
axes[2].bar(["Model A (v1)", "Model B (v2)"],
            [ab_results["model_a_avg_latency"], ab_results["model_b_avg_latency"]],
            color=["steelblue", "coral"])
axes[2].set_title("Average Latency (ms)")

plt.tight_layout()
plt.show()

## 5. Data Drift Monitoring

In [ ]:
# Check drift status after sending predictions
drift = api("get", "/drift/status")
print("Drift Status:", json.dumps(drift, indent=2))

## 6. Model Rollback

In [ ]:
# Switch to v2, then rollback
print("Activating v2...")
api("post", "/model/activate", json={"version": "v2"})
print(f"Active: {api('get', '/health')['active_model']}")

print("\nRolling back...")
result = api("post", "/model/rollback")
print(f"Rolled back to: {result['version']}")
print(f"Active: {api('get', '/health')['active_model']}")

## 7. Prometheus Metrics

View raw metrics or check the Grafana dashboard at http://localhost:3000

In [ ]:
# Fetch raw Prometheus metrics
resp = requests.get(f"{BASE_URL}/metrics")
# Show first 50 lines
lines = resp.text.strip().split("\n")
print(f"Total metric lines: {len(lines)}")
print("\nSample metrics:")
for line in lines[:30]:
    print(line)

---

## Next Steps

1. **Grafana Dashboard**: Open http://localhost:3000 to see real-time monitoring
2. **Load Test**: Run `bash scripts/load_test.sh` to generate traffic
3. **PyTorch Model**: Run `python examples/deploy_torch_model.py` to add a PyTorch model
4. **Custom Model**: Register your own model using the `/model/register` API